# 🎬 EscenIA · Generador de imágenes (versión fácil, 1 botón)

**Solo 2 pasos:**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4) → Guardar**.
2. Pulsa el botón **▶** de la única celda de abajo. (Si sale "este cuaderno no lo creó Google", dale **Ejecutar de todos modos**.)

Luego espera: instala todo, prepara el modelo y crea las imágenes de tu historia. Al final te descarga un **.zip**. La primera vez tarda varios minutos (descarga el modelo). ¡Eso es todo!

> Para cambiar la historia o el personaje, edita las líneas `historia = "..."` y `personaje = "..."` dentro de la celda.


In [ ]:
#@title ▶ Pulsa aquí para crear tu video (imágenes) { display-mode: "form" }

# ====== ESCRIBE TU HISTORIA AQUÍ ======
historia  = "Había una vez un pequeño dinosaurio llamado Rex. Vivía en un valle lleno de flores brillantes y cascadas de colores. Un día, Rex descubrió una cueva misteriosa escondida entre las rocas. Dentro brillaba un cristal mágico con todos los colores del arcoíris. Rex supo que esa noche comenzaría la aventura más grande de su vida."
personaje = "un pequeño dinosaurio verde de ojos grandes y amigables"
estilo    = "Animación 3D (Pixar)"   #@param ["Animación 3D (Pixar)", "Anime / Ghibli", "Acuarela infantil", "Realista cine", "Cómic"]
formato   = "Vertical (Shorts/TikTok)"  #@param ["Vertical (Shorts/TikTok)", "Horizontal (YouTube)", "Cuadrado"]
# ======================================

import subprocess, sys
print("⏳ Preparando el programa (1-2 min)...")
subprocess.run([sys.executable,"-m","pip","-q","install","diffusers","transformers","accelerate","safetensors","--upgrade"])

import torch, re, os, zipfile
from diffusers import StableDiffusionXLPipeline
from IPython.display import display

if not torch.cuda.is_available():
    raise SystemExit("❌ No hay GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU (T4) → Guardar, y vuelve a pulsar ▶.")
print("✅ GPU:", torch.cuda.get_device_name(0))

print("⏳ Cargando el modelo (la primera vez descarga ~7 GB)...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True
).to("cuda")
for _m in ("enable_attention_slicing","enable_vae_slicing"):
    try: getattr(pipe,_m)()
    except Exception: pass
print("✅ Modelo listo")

ESTILOS = {
 "Animación 3D (Pixar)":"Pixar-style 3D animated movie still, cinematic lighting, highly detailed, vibrant colors, soft global illumination, depth of field, charming character design",
 "Anime / Ghibli":"anime style, Studio Ghibli inspired, soft painterly background, cel shading, warm cinematic lighting",
 "Acuarela infantil":"children book watercolor illustration, soft pastel colors, hand painted, storybook style",
 "Realista cine":"cinematic photography, ultra realistic, dramatic lighting, 8k, film still",
 "Cómic":"comic book illustration, bold ink lines, dynamic shading, graphic novel style",
}
DIMS = {"Vertical (Shorts/TikTok)":(832,1216),"Horizontal (YouTube)":(1216,832),"Cuadrado":(1024,1024)}
NEG = "blurry, low quality, deformed, extra limbs, text, watermark, ugly, bad anatomy"

def trocear(t):
    parts=re.split(r"(?<=[\.\!\?…])\s+", re.sub(r"\s+"," ",t).strip())
    out=[]; buf=""
    for p in parts:
        p=p.strip()
        if not p: continue
        if len((buf+" "+p).strip())<45: buf=(buf+" "+p).strip()
        else:
            if buf: out.append(buf)
            buf=p
    if buf: out.append(buf)
    return out[:12]

frases=trocear(historia); w,h=DIMS[formato]; est=ESTILOS[estilo]
os.makedirs("/content/escenas",exist_ok=True)
for f in os.listdir("/content/escenas"): os.remove("/content/escenas/"+f)
print(f"\n🎬 Creando {len(frases)} escenas...\n")
gen=torch.Generator("cuda"); rutas=[]
for i,fr in enumerate(frases):
    gen.manual_seed(100+i*7)
    img=pipe(prompt=f"{fr}. Featuring {personaje}. {est}", negative_prompt=NEG,
             width=w,height=h,num_inference_steps=30,guidance_scale=7.0,generator=gen).images[0]
    r=f"/content/escenas/escena_{i+1:02d}.png"; img.save(r); rutas.append(r)
    print(f"  ✓ Escena {i+1}: {fr[:55]}...")
    display(img.resize((w//3,h//3)))

zp="/content/mis_escenas.zip"
with zipfile.ZipFile(zp,"w") as z:
    for r in rutas: z.write(r,os.path.basename(r))
print("\n✅ ¡Listo! Descargando mis_escenas.zip ...")
from google.colab import files; files.download(zp)
